In [ ]:
# Imports
from pyaesa import set_workspace, download_pop_gdp, download_ar6
from pyaesa import process_pop_gdp, process_ar6
from pyaesa import prepare_external_inputs
from pyaesa import deterministic_acc

# Notebook 01 — Allocating Carbon Budgets (aCC)

This notebook uses **pyaesa** to compute per-country allocated carrying capacities (aCC) — each country's fair share of the global carbon budget under the IPCC AR6 pathways.

**Method:** Equal per capita allocation (EG) — every person on Earth gets the same share of the remaining budget.  
**Data sources:** IPCC AR6 climate pathways · World Bank population data  
**Output:** `pacific_dataviz/B2_acc/` — aCC values per country per year (2000–2050)

In [2]:
# 1. Initialize workspace (always first, every session)
set_workspace(top_path="/Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026")

Workspace setup guidance information is available in:
/Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/data_raw/summary.log


## Step 1 — Workspace setup

Point pyaesa to the project root. All downloads and outputs go into subfolders here.

In [3]:
download_pop_gdp()

## Step 2 — Download raw data

One-time downloads. Safe to re-run (pyaesa skips if already present).
- `download_pop_gdp()`: World Bank population + GDP, 1995–2024 (~1 MB)
- `download_ar6()`: IPCC AR6 climate pathways (~210 MB)

In [4]:
download_ar6()

In [6]:
prepare_external_inputs(project_name="pacific_dataviz")

[prepare_external_inputs] External input guidance and examples imported

External aSoCC input folders:
/Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/pacific_dataviz/B1_asocc/ext_asocc
  -> deterministic/: runnable deterministic examples and project external aSoCC files.
  -> monte_carlo/: runnable Monte Carlo examples and project external aSoCC runs.

External aSoCC runnable examples:
/Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/pacific_dataviz/B1_asocc/ext_asocc
  -> deterministic/CO(S).csv: deterministic one step example for base years.
  -> deterministic/CO(S)_ssp2.csv: deterministic one step SSP2 example.
  -> deterministic/l1_AR(E)_l2_UT(S)__ef_3.1.csv: deterministic two step EF 3.1 example.
  -> monte_carlo/CO(S).csv: normal CSV Monte Carlo one step example.
  -> monte_carlo/CO(S)/: compact CSV Monte Carlo one step example.
  -> monte_carlo/l1_AR(E)_l2_UT(S)__ef_3.1.csv: normal CSV Monte Carlo two step EF 3.

[prepare_external_inputs] Summary:
  Output folder:
  /Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/pacific_dataviz
  Output folders:
  logs: summary log.

  Phase A: LCA:
    ext_lca:
      Input folder:
      /Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/pacific_dataviz/A_lca/ext_lca
      Output folders:
      templates: external LCA README guidance.
      deterministic: deterministic external LCA examples.
      monte_carlo: Monte Carlo external LCA examples.

  Phase B.1: aSoCC:
    ext_asocc:
      Input folder:
      /Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/pacific_dataviz/B1_asocc/ext_asocc
      Output folders:
      templates: external aSoCC README guidance.
      deterministic: deterministic external aSoCC examples.
      monte_carlo: Monte Carlo external aSoCC examples.

## Step 3 — Process raw data

Transforms raw downloads into pyaesa's internal format. AR6 processing covers 2000–2050 with harmonization against historical baselines.

In [7]:
process_pop_gdp()

In [8]:
process_ar6(years=range(2000, 2051))

[process_ar6] Summary:
  Run status: reused
  Study period: 2000-2050
  AR6 categories: C1-C4
  SSP scenarios: 1-5
  Harmonization: True
  Harmonization method: offset

  Phase B.0: Dynamic AR6 CC:
    process_ar6:
      Processed pathway coverage:
        Emissions(net)|Kyoto Gases|WO AFOLU: 142 model-scenario pairs
        Emissions(net)|Kyoto Gases: 142 model-scenario pairs
        Emissions(net)|CO2|WO AFOLU: 142 model-scenario pairs
        Emissions(net)|CO2: 142 model-scenario pairs
        Emissions(gross)|Kyoto Gases|WO AFOLU: 142 model-scenario pairs
        Emissions(gross)|Kyoto Gases: 142 model-scenario pairs
        Emissions(gross)|CO2|WO AFOLU: 142 model-scenario pairs
        Emissions(gross)|CO2: 142 model-scenario pairs
        Emissions(gross_alt)|Kyoto Gases|WO AFOLU: 110 model-scenario pairs
        Emissions(gross_alt)|Kyoto Gases: 97 model-scenario pairs
        Emissions(gross_alt)|CO2|WO AFOLU: 101 model-scenario pairs
        Emissions(gross_alt)|CO2: 59 mode

In [9]:
result = deterministic_acc(
    project_name="pacific_dataviz",
    source="iso3",
    fu_code="L1.a",
    years=range(2000, 2051),
    lcia_method="gwp100_lcia",
    base_asocc_args={
        "include_lcia_based_allocation_methods": False,
    },
)

## Step 4 — Compute allocated carrying capacities (aCC)

`deterministic_acc()` computes each country's fair share of the global carbon budget.

- `source="iso3"`: uses World Bank country list — covers all Pacific islands individually
- `fu_code="L1.a"`: total national consumption boundary
- `lcia_method="gwp100_lcia"`: GWP100 characterization (CO2-eq, AR6 factors)
- `include_lcia_based_allocation_methods=False`: required for iso3 source (no EXIOBASE needed)

Output: aCC in kg CO2-eq per country per year, stored in `pacific_dataviz/B2_acc/`